# Task 1: GPT-2 Fine-Tuning for Recipe Generation with LoRA and Quantization

This notebook implements Task 1 of the project requirements. We will fine-tune a GPT-2 model to generate cooking recipes from a title and a list of ingredients.

To make the process memory-efficient, we will use:
1.  **Quantization:** Loading the model in 4-bit using the `bitsandbytes` library.
2.  **LoRA (Low-Rank Adaptation):** Using the `peft` library to only train a small number of adapter weights instead of the entire model.

## 1. Setup

Install the necessary libraries for this task.

In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.4 MB/s eta 0:00:00


In [2]:
!pip install opendatasets

In [3]:
import opendatasets as od
od.download("https://www.kaggle.com/datasets/nazmussakibrupol/3a2mext/data")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: f223644saifullah
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/nazmussakibrupol/3a2mext


100%|██████████| 491M/491M [00:01<00:00, 470MB/s]


## 2. Load Dataset

We'll use the `recipe_nlg` dataset, which is the same as the one specified in the project requirements, but available directly on the Hugging Face Hub.

In [4]:
!pip install pandas
import pandas as pd
from datasets import Dataset
import ast

# Load your CSV file
# Make sure 'recipes.csv' is in the same directory as this notebook
df = pd.read_csv('/content/3a2mext/3A2M_EXTENDED.csv')

# Reduce the dataset size to prevent memory issues by taking a random sample
if len(df) > 50000:
    df = df.sample(n=50000, random_state=42).reset_index(drop=True)
    print(f"Using a random sample of {len(df)} recipes to save memory.")

# The NER and directions columns look like string representations of lists.
# We need to parse them into actual lists.
def parse_string_list(s):
    try:
        # Ensure the input is a string before trying to evaluate
        if isinstance(s, str):
            return ast.literal_eval(s)
        return [] # Return empty list if not a string
    except (ValueError, SyntaxError):
        # Return an empty list if parsing fails
        return []

df['NER'] = df['NER'].apply(parse_string_list)
df['directions'] = df['directions'].apply(parse_string_list)

# Convert the pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(df)

print("Dataset loaded and converted:")
print(dataset[0])

Using a random sample of 50000 recipes to save memory.
Dataset loaded and converted:
{'title': 'Jamaican Sweet Potato Pone', 'NER': ['sweet potatoes', 'brown sugar', 'butter', 'orange juice', 'eggs', 'dark rum', 'kosher salt'], 'Extended_NER': "['30 minutes', 'eggs', 'brown sugar', '20 minutes', 'Stir in the orange juice', 'sweet potatoes', 'kosher salt', '350\\\\u00b0F.', 'Lightly beat the egg yolks', 'dark rum', 'butter', 'Bake', 'orange juice', 'Cook']", 'genre': 'vegetables', 'label': 4, 'directions': ['Preheat the oven to 350°F.', 'Cook the sweet potato slices in boiling water for 20 minutes or until tender.', 'Drain and mash with the brown sugar and butter.', 'Stir in the orange juice.', 'Lightly beat the egg yolks and add them to the mixture with the rum.', 'Grease a pie plate.', 'Beat the egg whites with the salt until stiff, but not dry.', 'Gently fold the egg whites into the potato mix.', 'Pour into the prpared pie plate.', 'Bake for 30 minutes or until a knife inserted in th

### Sub-sampling the Dataset to Prevent Memory Issues

A kernel crash is often caused by running out of RAM. Since the full dataset is very large (over 2 million rows), processing it all at once can exceed the memory limits of a free Kaggle environment.

To prevent this, we will work with a smaller, random sample of the data (e.g., 50,000 rows). This will significantly reduce memory consumption while still providing a diverse dataset for fine-tuning.

## 3. Load Model and Tokenizer

We will load the `gpt2` model and its tokenizer. To enable efficient training, we'll use `BitsAndBytesConfig` to load the model in 4-bit precision.

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "gpt2"

# Configure quantization to load the model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model with the specified quantization configuration
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True
)
model.config.use_cache = False # Disable cache for training

# Load the tokenizer and set the padding token
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## 4. Data Preprocessing and Tokenization

We'll format the dataset into a single text string for each recipe and then tokenize it.

In [6]:
# Filter out rows with missing values in critical columns
# A recipe is not useful without a title, ingredients, or directions
original_size = len(dataset)
dataset = dataset.filter(lambda example: example['title'] is not None and example['NER'] and example['directions'])
new_size = len(dataset)
print(f"Filtered out {original_size - new_size} rows with missing data.")

def format_recipe(example):
    # Clean up title if it has leading/trailing whitespace
    title = example['title'].strip()

    # Join the list of ingredients and directions into single strings
    ingredients = ', '.join(example['NER'])
    directions = ' '.join(example['directions'])

    # Combine title, ingredients, and directions into a single string
    # The eos_token acts as a separator, signaling the end of a recipe
    return {
        "text": f"Title: {title}\nIngredients: {ingredients}\nDirections: {directions}{tokenizer.eos_token}"
    }

formatted_dataset = dataset.map(format_recipe)

def tokenize_function(examples):
    # Tokenize the formatted text
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = formatted_dataset.map(tokenize_function, batched=True)

# We only need the input_ids, attention_mask, and labels for training
# Adjust columns to remove based on your new dataset
tokenized_dataset = tokenized_dataset.remove_columns(dataset.column_names)
tokenized_dataset.set_format("torch")

# For Causal LM, the labels are the same as the input_ids
tokenized_dataset = tokenized_dataset.map(lambda examples: {'labels': examples['input_ids']}, batched=True)

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

Filtered out 6042 rows with missing data.


Map:   0%|          | 0/43958 [00:00<?, ? examples/s]

Map:   0%|          | 0/43958 [00:00<?, ? examples/s]

Map:   0%|          | 0/43958 [00:00<?, ? examples/s]

## 5. Configure LoRA

Now, we set up the LoRA configuration using PEFT (Parameter-Efficient Fine-Tuning).

In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for k-bit training
model = prepare_model_for_kbit_training(model)

# Define the LoRA configuration
lora_config = LoraConfig(
    r=16, # Rank of the update matrices
    lora_alpha=32, # Alpha parameter for scaling
    target_modules=["c_attn"], # Target modules to apply LoRA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Get the PEFT model by wrapping the base model with LoRA config
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

trainable params: 589,824 || all params: 125,029,632 || trainable%: 0.4717


## 6. Fine-Tuning the Model

We'll use the `Trainer` class from the `transformers` library to fine-tune the model.

In [ ]:
import transformers
from pathlib import Path

# Check if model already exists
model_save_path = "gpt2-recipe-lora-adapter"
if Path(model_save_path).exists():
    print(f"✅ Model already exists at '{model_save_path}'")
    print("Loading the existing model instead of training...")
    
    from peft import PeftModel
    
    # Load the base model with quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        trust_remote_code=True
    )
    base_model.config.use_cache = False
    
    # Load the PEFT model with saved adapter
    peft_model = PeftModel.from_pretrained(base_model, model_save_path)
    print("✅ Model loaded successfully!")
    
else:
    print(f"⚙️ Model not found. Starting training...")
    
    # Define training arguments
    training_args = transformers.TrainingArguments(
        output_dir="gpt2-recipe-generator",
        num_train_epochs=3, # As suggested, 3-5 epochs are sufficient
        per_device_train_batch_size=2, # Reduce batch size to fit in memory
        gradient_accumulation_steps=4, # Compensate for small batch size
        learning_rate=2e-4,
        fp16=True, # Enable mixed precision training
        logging_dir='./logs',
        logging_steps=50,
        save_strategy="epoch",
    )

    # Create the Trainer
    trainer = transformers.Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    # Start training
    trainer.train()
    
    # Save the trained LoRA adapter
    peft_model.save_pretrained(model_save_path)
    print(f"✅ Model trained and saved to '{model_save_path}'")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: su3561116 (su3561116-wqxr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.004700
100,2.819100
150,2.728900
200,2.753900
250,2.760000
300,2.722900
350,2.698000
400,2.726200
450,2.670500
500,2.680900


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


## 7. Save the Model and Generate Recipes

After training, we save the LoRA adapter and use the fine-tuned model for inference.

In [ ]:
# Save the trained LoRA adapter
peft_model.save_pretrained("gpt2-recipe-lora-adapter")

# Prepare the model for inference
device = "cuda" if torch.cuda.is_available() else "cpu"

prompt_text = "Title: Classic Pancakes\nIngredients: Flour, Egg, Milk, Sugar\nDirections:"
inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

# Generate recipe
peft_model.eval()
with torch.no_grad():
    outputs = peft_model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=200,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

generated_recipe = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- Generated Recipe ---")
print(generated_recipe)

## 8. (Optional) Merge LoRA with Base Model

If you want to create a standalone fine-tuned model, you can merge the LoRA weights with the base model and save it.

In [ ]:
# from peft import PeftModel

# # Reload the base model
# base_model = AutoModelForCausalLM.from_pretrained(model_name)

# # Load the PEFT model with LoRA adapter
# merged_model = PeftModel.from_pretrained(base_model, "gpt2-recipe-lora-adapter")

# # Merge the weights
# merged_model = merged_model.merge_and_unload()

# # Save the merged model
# merged_model.save_pretrained("gpt2-recipe-finetuned-merged")
# tokenizer.save_pretrained("gpt2-recipe-finetuned-merged")